# The Shell and Its Mechanics

## Keyboard shortcuts — the ones that pay back instantly

The shell's command line is editable like a tiny text editor. Most beginners arrow-key character-by-character; pros use these:

| Shortcut | Action |
|---|---|
| **Ctrl-A** | jump to start of line |
| **Ctrl-E** | jump to end of line |
| **Ctrl-U** | delete everything from cursor to start of line |
| **Ctrl-K** | delete everything from cursor to end of line |
| **Ctrl-W** | delete the word before the cursor |
| **Ctrl-Y** | paste back what you just deleted with U/K/W |
| **Alt-B** | jump back one word |
| **Alt-F** | jump forward one word |
| **Ctrl-L** | clear the screen (same as `clear` but instant) |
| **Ctrl-R** | reverse-search through history (you met this in 01) |
| **Ctrl-C** | abandon current line (or interrupt the running command) |
| **Ctrl-D** | end-of-input — on an empty prompt, logs you out of the shell |
| **Alt-.** (period) | paste the last argument of the previous command |

The last one — **Alt-.** — is magical. After `mkdir /tmp/long/path/to/work`, the next command `cd <Alt-.>` becomes `cd /tmp/long/path/to/work` automatically. Tap it multiple times to walk back through earlier last-arguments.

These shortcuts are **Emacs-style**, which is bash's default editing mode. If you ever change to Vi mode (`set -o vi`), the key bindings change — but Emacs mode is the default and what almost everyone uses.

Practice them in your shell now. Spending five minutes doing this once saves you hours over a career.

## Variables — naming values

A **variable** in the shell is a label you give to a value, so you can refer to it later without retyping. Think of it as a sticky note: you write a value on it, give the note a name, and stick it on the wall.

To create one: `name=value`. **No spaces around the `=`.** That's the most common beginner mistake — `name = value` is interpreted as a command called `name`.

To read it back, put a **`$`** in front of the name: `$name`. The shell substitutes the value.

```
greeting="Hello, Linux"
echo $greeting           # prints: Hello, Linux
```

Two things to know straight away:

1. **No types.** Everything is a string. `count=5` stores the *string* "5", not the number 5.
2. **Naming convention.** Lowercase for your own variables (`user_input`, `filename`), UPPERCASE for environment variables (`HOME`, `PATH`). This isn't enforced by the shell — it's a convention, but a strong one.

To unset a variable, use **`unset name`** (note: no `$` here — you're naming the variable, not reading its value).

To see all variables currently defined, run **`set`** (it shows a lot — pipe to `head` or `grep`).

In [ ]:
!greeting="Hello, Linux" && echo $greeting
!count=5 && echo "I have $count files"
!echo $undefined_variable
!unset greeting && echo "greeting is now: '$greeting'"

## The environment — variables that programs inherit

There are actually **two** kinds of variables in the shell:

- **Shell variables** — only visible inside the current shell. The `greeting` you just made is one of these.
- **Environment variables** — visible to every program the shell launches. These get *inherited* by child processes.

The environment is a list of `NAME=VALUE` pairs that gets passed to every command you run. It's how programs find out things like your home directory, your username, where to find executables, your preferred editor, and so on.

To see the full environment, use **`env`** or **`printenv`**. They print one variable per line.

To see just one variable: `echo $VARIABLE` or `printenv VARIABLE`.

The most important environment variables — the ones you'll see and edit constantly:

| Variable | What it holds |
|---|---|
| **`HOME`** | path to your home directory (e.g. `/home/ubuntu`). What `~` expands to. |
| **`USER`** | your username. Same value as `whoami` prints. |
| **`SHELL`** | the path to your login shell (e.g. `/bin/bash`). **Not necessarily the shell you're using right now** — it's just your *default*. |
| **`PATH`** | colon-separated list of directories the shell searches when you type a command. |
| **`PWD`** | the current directory (kept up to date automatically by `cd`). |
| **`LANG`** | your locale (e.g. `en_GB.UTF-8`). Affects sort order, date formats, error messages. |
| **`EDITOR`** | which editor commands like `crontab -e` and `git commit` should launch. Often set to `vim`, `nano`, `emacs`, or `code`. |
| **`TERM`** | what kind of terminal you're using (`xterm-256color`, `screen`, `tmux-256color`). Tools use this to decide what colours and features to use. |

**`PATH` deserves special attention.** When you type `ls`, the shell doesn't know where `ls` lives on disk. It looks in each directory in `PATH`, in order, and runs the first match. If `PATH` is `/usr/local/bin:/usr/bin:/bin`, it looks in `/usr/local/bin` first, then `/usr/bin`, then `/bin`. The command **`which ls`** tells you which one it would run; **`type ls`** is more thorough (also reveals aliases and shell built-ins).

In [ ]:
!echo "HOME=$HOME"
!echo "USER=$USER"
!echo "SHELL=$SHELL"
!echo "PATH=$PATH"
!env | head -10
!which ls
!type ls

## `export` — promoting a shell variable to the environment

A plain assignment (`name=value`) creates a **shell variable** — only the current shell sees it. To make a variable visible to commands you launch, use **`export`**:

```bash
MY_VAR="hello"           # shell variable, current shell only
export MY_VAR            # now also in the environment

# or both in one line:
export MY_VAR="hello"
```

You can verify: a shell variable shows up in `set` but not in `env`. An exported (environment) variable shows up in both.

When you launch a new program, it gets a *copy* of the environment. Changes the child makes to its own environment never affect the parent. This is why setting a variable inside a script doesn't persist when the script exits — the script's shell exited, taking its environment with it.

This becomes very relevant for shell configuration. Files like `~/.bashrc` and `~/.profile` are read by the shell at startup and typically `export` variables you want to persist across all your sessions. We'll cover those files in detail in notebook 06.

In [ ]:
!FOO=local_only && echo "in this shell: $FOO" && bash -c 'echo "child sees: $FOO"'
!export FOO=now_exported && bash -c 'echo "child now sees: $FOO"'

## Quoting — when the shell takes you literally

Type this:

```bash
echo Hello   World
```

The output is `Hello World` — **one space**, not three. That's because the shell collapses runs of whitespace between arguments before passing them to `echo`. To keep the multiple spaces, you have to **quote**:

```bash
echo "Hello   World"
```

Quoting is the shell's way of letting you say "treat this string literally" — to one degree or another. There are **three forms of quoting**, each switching off a different amount of the shell's normal interpretation.

**Single quotes** (`'...'`) — everything inside is literal. No variable substitution, no command substitution, no nothing. The string between single quotes goes to the command as-is.

```bash
echo 'My home is $HOME'        # prints: My home is $HOME
```

**Double quotes** (`"..."`) — preserve whitespace and most special characters, **but variable substitution and command substitution still happen**.

```bash
echo "My home is $HOME"        # prints: My home is /home/ubuntu
```

**Backslash** (`\`) — escapes a single following character. `\$` is a literal `$`, `\ ` is a literal space inside an unquoted word, `\n` inside double quotes is... still literally `\n` in echo (bash doesn't interpret backslash escapes in `echo` unless you use `echo -e` or `printf`).

The practical rule: **when in doubt, use double quotes around variable references** — `"$var"`. This is by far the most common shell bug for beginners: an unquoted variable that contains spaces gets split into multiple arguments. Quoting it preserves it as one piece.

In [ ]:
!echo Hello    World
!echo "Hello    World"
!echo 'My home is $HOME and today is $(date)'
!echo "My home is $HOME and today is $(date)"
!name="John Smith" && ls "/tmp/$name" 2>/dev/null; ls /tmp/$name 2>/dev/null

## Output redirection — pointing the firehose

By default, commands write their output to your terminal. **Redirection** sends that output somewhere else — usually a file. The redirection operators:

| Operator | Effect |
|---|---|
| **`> file`** | write output to `file`, **overwriting** any existing content |
| **`>> file`** | write output to `file`, **appending** to it |
| **`2> file`** | write **error messages** to `file` |
| **`&> file`** | write both normal output and errors to `file` |
| **`2>&1`** | merge errors into wherever the normal output is going |
| **`> /dev/null`** | throw output away (it goes to the "bit bucket") |

Two streams to understand:

- **Standard output (stdout)** — the normal output of a successful command. Numbered as file descriptor `1`.
- **Standard error (stderr)** — error messages and diagnostics. Numbered as file descriptor `2`.

These are kept separate so you can redirect one without the other. If `ls /etc /nonexistent > out.txt`, then `out.txt` gets the listing of `/etc` and the error about `/nonexistent` still appears on your terminal — because only stdout was redirected.

To silence a command entirely: `command > /dev/null 2>&1`. To capture both into one log: `command &> output.log`.

**Order matters when combining.** Read redirections left-to-right. `command > file 2>&1` means "redirect stdout to file, then redirect stderr to wherever stdout goes (which is now the file)." But `command 2>&1 > file` means "redirect stderr to wherever stdout goes (still the terminal at this point), then redirect stdout to file" — you've made stderr go to your terminal and stdout go to the file. Probably not what you wanted.

In [ ]:
!echo "first line" > /tmp/demo.txt
!echo "second line" >> /tmp/demo.txt
!cat /tmp/demo.txt
!ls /etc /nonexistent > /tmp/out.txt 2> /tmp/err.txt; echo "--- stdout:"; head -3 /tmp/out.txt; echo "--- stderr:"; cat /tmp/err.txt
!ls /etc /nonexistent &> /tmp/all.txt; echo "--- combined:"; head -5 /tmp/all.txt

## Input redirection and here-documents

Most commands also accept input from a file or directly from your command line. Three forms:

**`< file`** — feed the contents of `file` as input to the command. Same effect as `cat file | command` but without the pipe.

```bash
wc -l < /etc/passwd
```

**Here-document** (`<< MARKER ... MARKER`) — embed a multi-line block of text directly in your command. Useful for sending several lines to a command without making a separate file.

```bash
cat <<EOF
line one
line two
line three
EOF
```

The word after `<<` (`EOF` by convention) is the marker that ends the block — pick anything that won't appear in your text. Variables inside a here-doc are expanded as usual (use `<<'EOF'` with quotes to disable expansion).

**Here-string** (`<<< "string"`) — a one-line version of a here-doc.

```bash
grep -E '^root' <<< "root:x:0:0::/root:/bin/bash"
```

Here-strings and here-docs are useful for testing commands without creating temporary files, and for sending data into commands that only accept stdin (like `mailx`, `psql`, `bc`).

In [ ]:
!wc -l < /etc/passwd
!cat <<EOF
first
second
$USER lives at $HOME
EOF
!grep -c bash <<< "$(cat /etc/passwd)"

## Pipes — the most important character in the shell

A pipe (`|`) takes the **stdout of one command** and feeds it directly into the **stdin of the next**. It's the operator that turns Linux's small tools into something more powerful than the sum of its parts.

```bash
ls /etc | wc -l
```

This runs `ls /etc`, but instead of printing to your terminal, the output goes to `wc -l`, which counts lines. The result: the number of entries in `/etc`.

You can chain as many as you want:

```bash
ls /etc | sort | head -5
ps -ef | grep bash | wc -l
cat /etc/passwd | cut -d: -f1 | sort
```

Each `|` connects the **stdout** of the left side to the **stdin** of the right side. **Stderr is not piped** — it still goes to your terminal unless you redirect it separately.

The Unix philosophy is built on pipes: write small programs that each do one thing well, accept input on stdin, and write output on stdout. Then compose them with pipes to solve bigger problems. You'll see this pattern over and over in the next notebook when we meet `grep`, `sed`, `awk`, `cut`, `sort`, and `uniq`.

A useful exit-code rule to know: by default, the exit status of a pipeline is the **last** command's exit status. If `false | true`, the pipeline succeeds because `true` succeeded. If you want a pipeline to fail when *any* command fails, you can `set -o pipefail` at the top of a script. (We come back to this in notebook 10.)

In [ ]:
!ls /etc | wc -l
!ls /etc | sort | head -5
!ps -ef | head -5
!cat /etc/passwd | cut -d: -f1 | head -5

## Globs — matching filenames with patterns

A **glob** is a wildcard pattern the shell uses to match filenames. You type `*.txt` and the shell turns it into the list of every `.txt` file in the current directory **before** your command sees it.

The pattern characters:

| Pattern | Matches |
|---|---|
| `*` | any number of characters (including zero), **except a leading `.`** |
| `?` | exactly one character |
| `[abc]` | one character — any of `a`, `b`, or `c` |
| `[a-z]` | one character — anything in the range `a` through `z` |
| `[!abc]` | one character — anything **except** `a`, `b`, or `c` |

A few examples:

- `ls *.txt` — all files ending in `.txt`
- `ls report?.csv` — `report1.csv`, `report9.csv`, but not `report10.csv`
- `ls [Ll]inux*` — files starting with `Linux` or `linux`
- `rm tmp[0-9]` — `tmp0` through `tmp9`

Two important rules:

1. **Globs are expanded by the shell, not by the command.** When you type `ls *.txt`, the shell turns `*.txt` into `a.txt b.txt c.txt` and runs `ls a.txt b.txt c.txt`. `ls` itself has no idea a glob was involved.
2. **If a glob matches no files, by default bash passes the literal pattern through.** So `ls *.notreal` becomes `ls *.notreal` — and `ls` complains "No such file or directory: *.notreal". This trips up beginners. The fix is to handle the no-match case in your scripts, or set `shopt -s nullglob` so non-matching globs expand to nothing.

**Globs are NOT regular expressions.** This is the most confusing thing for newcomers. `*` in a glob means "any number of characters"; `*` in a regular expression means "zero or more of the previous character." Different languages, different rules, often used near each other. Regular expressions show up in `grep`, `sed`, and `awk` — covered in notebook 04.

In [ ]:
!ls /etc/*.conf 2>/dev/null | head -5
!ls /etc/host* 2>/dev/null
!ls /etc/[a-c]* 2>/dev/null | head -5

## Expansion order — what the shell does to your line

Here's the rough sequence of things the shell does to every line you type, **before** any command runs:

1. **History expansion** — `!!`, `!42`, `!ls` are replaced with previous commands.
2. **Brace expansion** — `{a,b,c}` becomes three arguments `a b c`; `{1..5}` becomes `1 2 3 4 5`.
3. **Tilde expansion** — `~` becomes your home directory; `~user` becomes that user's home.
4. **Variable expansion** — `$name` is replaced with the variable's value.
5. **Command substitution** — `$(command)` runs `command` and inserts its output.
6. **Arithmetic expansion** — `$(( 2 + 3 ))` evaluates to `5`.
7. **Word splitting** — the result is split on whitespace into separate arguments.
8. **Glob expansion** — `*`, `?`, `[abc]` are matched against filenames.
9. **Quote removal** — surviving `'`, `"`, `\` are removed before the command runs.

You don't need to memorise all of this right now. The important thing is to know **it happens in this order**, because some surprising bugs come from the order being wrong-for-what-you-expected.

For example: `name=$(ls *.txt)` — first `*.txt` is glob-expanded by the shell (turning into a list of files), then `ls` runs on that list, then `$(...)` captures its output, then the result is assigned to `name`. If `name` then contains filenames with spaces, using `$name` later (without quotes) will split it on those spaces.

This is also why **quoting variables is the safe default**. `"$name"` skips word-splitting and glob-expansion on the value — exactly what you want when the value is meant to be one string.

We'll come back to expansion subtleties as they come up. For now, you've seen the big picture, and you know the order.

In [ ]:
!echo {a,b,c}.txt
!echo {1..5}
!echo "today is $(date +%A)"
!echo $((10 * 7))
!name="John Smith" && echo $name && echo "$name"